# Loading Data

In [1]:
import pandas as pd
import numpy as np

In [24]:
# train data
train = pd.read_csv("train.csv")
lgb_8 = pd.read_csv("oof_lgb.csv")
xgb_8 = pd.read_csv("oof_xgb.csv")
lgb_9 = pd.read_csv("oof_lgb_9.csv")
xgb_9 = pd.read_csv("oof_lgb_9.csv")
cat_10 = pd.read_csv("oof_cat_10.csv")
xgb_11 = pd.read_csv("oof_xgb_11.csv")
logit_12 = pd.read_csv("oof_logit_12.csv")

In [25]:
# train data
train_blend = train[['id','addicted_label']]
train_blend['lgb_8']=lgb_8['addicted_label']
train_blend['xgb_8']=xgb_8['addicted_label']
train_blend['lgb_9']=lgb_9['addicted_label']
train_blend['xgb_9']=xgb_9['addicted_label']
train_blend['cat_10']=cat_10['addicted_label']
train_blend['xgb_11']=xgb_11['addicted_label']
train_blend['logit_12']=logit_12['addicted_label']

In [26]:
train_blend.head()

,id,addicted_label,lgb_8,xgb_8,lgb_9,xgb_9,cat_10,xgb_11,logit_12
0,0,1,0.854469,0.825155,0.876218,0.876218,0.835556,0.597654,0.742463
1,1,0,0.023812,0.019690,0.033296,0.033296,0.037425,0.085933,0.367813
2,2,0,0.469616,0.378026,0.469198,0.469198,0.452888,0.317818,0.743042
3,3,1,0.379479,0.368591,0.349492,0.349492,0.325944,0.339354,0.521827
4,4,1,0.999983,0.999973,0.999984,0.999984,0.999793,0.999920,0.983964


In [27]:
# test data
test = pd.read_csv("test.csv")
sub_lgb_8 = pd.read_csv("sub_lgb.csv")
sub_xgb_8 = pd.read_csv("sub_xgb.csv")
sub_lgb_9 = pd.read_csv("sub_lgb_9.csv")
sub_xgb_9 = pd.read_csv("sub_lgb_9.csv")
sub_cat_10 = pd.read_csv("sub_cat_10.csv")
sub_xgb_11 = pd.read_csv("sub_xgb_11.csv")
sub_logit_12 = pd.read_csv("sub_logit_12.csv")

In [28]:
# test data
test_blend = test[['id']]
test_blend['lgb_8']=sub_lgb_8['addicted_label']
test_blend['xgb_8']=sub_xgb_8['addicted_label']
test_blend['lgb_9']=sub_lgb_9['addicted_label']
test_blend['xgb_9']=sub_xgb_9['addicted_label']
test_blend['cat_10']=sub_cat_10['addicted_label']
test_blend['xgb_11']=sub_xgb_11['addicted_label']
test_blend['logit_12']=sub_logit_12['addicted_label']

In [29]:
test_blend.head()

,id,lgb_8,xgb_8,lgb_9,xgb_9,cat_10,xgb_11,logit_12
0,691369,0.999411,0.999682,0.999372,0.999372,0.999349,0.998112,0.943226
1,691370,0.957203,0.897480,0.957812,0.957812,0.943218,0.838531,0.953712
2,691371,0.943774,0.935185,0.949821,0.949821,0.945361,0.985683,0.954538
3,691372,0.990545,0.992388,0.992187,0.992187,0.988822,0.987936,0.939049
4,691373,0.998186,0.998274,0.997824,0.997824,0.997532,0.994729,0.981143


In [30]:
# submission
sample_submit = pd.read_csv("sample_submission.csv")

# Modeling Preparation

In [31]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [32]:
'''
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
'''

'\nimport lightgbm as lgb\nimport xgboost as xgb\nfrom catboost import CatBoostClassifier, Pool\n'

In [34]:
train_blend.dtypes

id                  int64
addicted_label      int64
lgb_8             float64
xgb_8             float64
lgb_9             float64
xgb_9             float64
cat_10            float64
xgb_11            float64
logit_12          float64
dtype: object

In [60]:
# 4. Separate features and target
# X = train_blend.drop(columns=["id","addicted_label","logit_12"])
X = train_blend.drop(columns=["id","addicted_label"])
y = train_blend["addicted_label"]
# X_test = test_blend.drop(columns=["id","logit_12"])
X_test = test_blend.drop(columns=["id"])

In [61]:
SEED = 42
N_FOLDS = 5
TARGET = 'addicted_label'

In [62]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

# Modeling

## Logistic Regression Model

In [49]:
from sklearn.linear_model import LogisticRegression

In [50]:
def fit_logit(X_tr, y_tr, X_va, y_va):
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [51]:
oof_blending, test_blending, auc_blending = run_cv(fit_logit, 'LogitModel')

  fold 0: AUC = 0.96356
  fold 1: AUC = 0.96414
  fold 2: AUC = 0.96443
  fold 3: AUC = 0.96513
  fold 4: AUC = 0.96414
LogitModel | OOF AUC = 0.96426 (mean 0.96428 ± 0.00051)


## Ensemble

In [52]:
# test_mean = (test_lgb + test_xgb + test_cat) / 3
# test_mean = (test_lgb + test_xgb ) / 2
# test_rank = (rankdata(test_lgb) + rankdata(test_xgb) + rankdata(test_cat)) / (3 * len(test_lgb))

def save_sub(pred, name):
    sub = sample_submit.copy()
    sub[TARGET] = pred
    sub.to_csv(name, index=False)
    print('guardado:', name)

# save_sub(test_lgb,  'sub_lgb.csv')
save_sub(test_blending,  'sub_blending_13_2.csv')
#save_sub(test_cat,  'sub_cat.csv')
# save_sub(test_mean, 'sub_ensemble_mean.csv')
# save_sub(test_rank, 'sub_ensemble_rank.csv')

guardado: sub_blending_13_2.csv


In [53]:
oof_blending

array([0.91860092, 0.03408436, 0.36957176, ..., 0.64719444, 0.9718048 ,
       0.23461181], shape=(691369,))

In [54]:
def save_oof(pred, name):
    oof = train.copy()
    oof = oof[["id",TARGET]]
    oof[TARGET] = pred
    oof.to_csv(name, index=False)
    print('saving:', name)

In [55]:
# save_oof(oof_lgb,'oof_lgb.csv')
save_oof(oof_blending,'oof_blending_13_2.csv')

saving: oof_blending_13_2.csv


## Blending with XGBoost

In [56]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [63]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [64]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.96569
  fold 1: AUC = 0.96627
  fold 2: AUC = 0.96643
  fold 3: AUC = 0.96714
  fold 4: AUC = 0.96598
XGBoost | OOF AUC = 0.96629 (mean 0.96630 ± 0.00049)


In [65]:
# save_sub(test_blending,  'sub_blending_13_3_nologit.csv')
# save_oof(oof_blending,'oof_blending_13_3_nologit.csv')

In [66]:
save_sub(test_blending,  'sub_blending_13_4_logit.csv')
save_oof(oof_blending,'oof_blending_13_4_logit.csv')

guardado: sub_blending_13_4_logit.csv
saving: oof_blending_13_4_logit.csv
